In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPEN_AI_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'

In [11]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5.4-mini')
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000017A4F363A90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000017A4F4BD550>, root_client=<openai.OpenAI object at 0x0000017A4DA674D0>, root_async_client=<openai.AsyncOpenAI object at 0x0000017A4F4BD090>, model_name='gpt-5.4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [35]:
import bs4

web_load = WebBaseLoader(web_path='https://en.wikipedia.org/wiki/Test_cricket', 
                         bs_kwargs=dict(parse_only=bs4.SoupStrainer(
                             'p' # to get only p (paragraph) tag only 
                            ),
                         ))
web_data = web_load.load()
web_data

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Test_cricket'}, page_content='\nTest cricket is a format of the sport of cricket, considered the game\'s most prestigious and traditional form. Often referred to as the "ultimate test" of a cricketer\'s skill, endurance and temperament, it is a first-class format of international cricket where two teams in whites, each representing their country, compete over a match that can last up to five days. It consists of up to four innings (up to two per team), with a minimum of ninety overs scheduled to be bowled in six hours per day, making it the sport with the longest playing time except for some multi-stage cycling races. A team wins the match by outscoring the opposition with the bat and bowling them out with the ball. Otherwise the match ends in a draw.It is contested by 12 teams which are the full-members of the International Cricket Council (ICC). The term "test match" was originally coined in 1861–62 but in a different conte

In [36]:
web_data[0].page_content

'\nTest cricket is a format of the sport of cricket, considered the game\'s most prestigious and traditional form. Often referred to as the "ultimate test" of a cricketer\'s skill, endurance and temperament, it is a first-class format of international cricket where two teams in whites, each representing their country, compete over a match that can last up to five days. It consists of up to four innings (up to two per team), with a minimum of ninety overs scheduled to be bowled in six hours per day, making it the sport with the longest playing time except for some multi-stage cycling races. A team wins the match by outscoring the opposition with the bat and bowling them out with the ball. Otherwise the match ends in a draw.It is contested by 12 teams which are the full-members of the International Cricket Council (ICC). The term "test match" was originally coined in 1861–62 but in a different context—that the English team was testing itself against all of the Australian colonies.[1][2] 

In [37]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_doc = text_splitter.split_documents(web_data)
split_doc

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Test_cricket'}, page_content='Test cricket is a format of the sport of cricket, considered the game\'s most prestigious and traditional form. Often referred to as the "ultimate test" of a cricketer\'s skill, endurance and temperament, it is a first-class format of international cricket where two teams in whites, each representing their country, compete over a match that can last up to five days. It consists of up to four innings (up to two per team), with a minimum of ninety overs scheduled to be bowled in six hours per day, making it the sport with the longest playing time except for some multi-stage cycling races. A team wins the match by outscoring the opposition with the bat and bowling them out with the ball. Otherwise the match ends in a draw.It is contested by 12 teams which are the full-members of the International Cricket Council (ICC). The term "test match" was originally coined in 1861–62 but in a different context

## Embedding and storing to DB

In [43]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

db = FAISS.from_documents(split_doc, embedding=embeddings)
db

In [49]:
db.similarity_search_with_score(query="What are the rules of test cricket")

[(Document(id='389bf4ad-4cb5-4bb1-a780-1485a104eb5d', metadata={'source': 'https://en.wikipedia.org/wiki/Test_cricket'}, page_content='Test cricket is a format of the sport of cricket, considered the game\'s most prestigious and traditional form. Often referred to as the "ultimate test" of a cricketer\'s skill, endurance and temperament, it is a first-class format of international cricket where two teams in whites, each representing their country, compete over a match that can last up to five days. It consists of up to four innings (up to two per team), with a minimum of ninety overs scheduled to be bowled in six hours per day, making it the sport with the longest playing time except for some multi-stage cycling races. A team wins the match by outscoring the opposition with the bat and bowling them out with the ball. Otherwise the match ends in a draw.It is contested by 12 teams which are the full-members of the International Cricket Council (ICC). The term "test match" was originally 

In [108]:
# document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question properly on the basis of the given context:
    {context}
    """
)

doc_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
doc_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question properly on the basis of the given context:\n    {context}\n    '), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000017A4F363A90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000017A4F4BD550>, root_client=<openai.OpenAI object at 0x0000017A4DA674D0>, root_async_client=<openai.AsyncOpenAI object at 0x0000017A4F4BD090>, model_name='gpt-5.4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)
| StrOutputParser()

In [105]:
from langchain_core.documents import Document

doc_chain.invoke({
    "input": "history of test cricket",
    "context": split_doc
})

'Test cricket is considered the **most prestigious and traditional** format of cricket, often called the **“ultimate test”** of a player’s skill, endurance, and temperament.\n\nKey points:\n- It is an **international first-class format** played by **two national teams**.\n- Matches usually last **up to five days**.\n- Each team can bat **twice**, so there are **up to four innings** in total.\n- A team wins by **scoring more runs than the opposition** and **bowling them out**; otherwise the match is a **draw**.\n- It is played by the ICC’s **12 full member teams**.\n- Historically, the first officially recognised Test match was played in **1877** between **Australia and England** at the Melbourne Cricket Ground.\n- Test cricket has evolved over time, including the introduction of **day/night Tests** and the **ICC World Test Championship**.'

### convert as retriever

In [109]:
vector_retriever = db.as_retriever()
vector_retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000017A525FF350>, search_kwargs={})

In [110]:
from langchain.chains import create_retrieval_chain

ret_chain = create_retrieval_chain(vector_retriever, doc_chain)
ret_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000017A525FF350>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question properly on the basis of the given context:\n    {context}\n    '), additional_kwargs={})])
            | ChatOpe

In [121]:
responses = []
queries = [
    "What is the history of Test cricket?",
    "Tell me the list countries whose team is playing test cricket? Tell their status",
    "When did India enter the world of test cricket?",
    "What are the main rules of test cricket?"
]

for query in queries:
    res = ret_chain.invoke({
        "input": query
    })
    responses.append(res)

In [122]:
for res in responses:
    print(f"Q: {res['input']}\n")
    print(f"A: {res['answer']}\n\n")

Q: What is the history of Test cricket?

A: The context describes **Test cricket** as the **most prestigious and traditional format of cricket**. It is an **international first-class format** played by **two teams in whites**, each representing a country, over a match that can last **up to five days**. A Test match can have **up to four innings** and is usually won by **outscoring the opponent and bowling them out**; otherwise, it ends in a **draw**. It is played by the **12 ICC full member teams**.


Q: Tell me the list countries whose team is playing test cricket? Tell their status

A: The passage is describing **Test status in cricket**.

So, the answer is: **Test status**.


Q: When did India enter the world of test cricket?

A: The context explains the history and scheduling changes of Test cricket:

- Tests are the highest level of cricket played between national teams with Test status.
- Because of concerns like matches potentially continuing indefinitely and the game’s perceive

In [123]:
res

{'input': 'What are the main rules of test cricket?',
 'context': [Document(id='389bf4ad-4cb5-4bb1-a780-1485a104eb5d', metadata={'source': 'https://en.wikipedia.org/wiki/Test_cricket'}, page_content='Test cricket is a format of the sport of cricket, considered the game\'s most prestigious and traditional form. Often referred to as the "ultimate test" of a cricketer\'s skill, endurance and temperament, it is a first-class format of international cricket where two teams in whites, each representing their country, compete over a match that can last up to five days. It consists of up to four innings (up to two per team), with a minimum of ninety overs scheduled to be bowled in six hours per day, making it the sport with the longest playing time except for some multi-stage cycling races. A team wins the match by outscoring the opposition with the bat and bowling them out with the ball. Otherwise the match ends in a draw.It is contested by 12 teams which are the full-members of the Internati